In [5]:
import os
import sys
import json
import random
import torch
from pathlib import Path

# 切换工作目录
os.chdir(r"E:\student\模型")
print(f"当前工作目录: {os.getcwd()}")

# 添加项目路径到 sys.path
sys.path.insert(0, r"E:\student\模型")
sys.path.insert(0, r"E:\student\模型\codebert")

# 导入模型和配置
from code_module.models.codebert_detector import CodeBertDetector
from common.config import TYPE_LABELS, LOCAL_MODEL_PATH

CHECKPOINT_PATH = Path(r"E:\student\模型\codebert\code_module\checkpoints\best.pt")
TEST_DATA_PATH = Path(r"E:\student\模型\codebert\code_module\data\test.json")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f" 配置完成，使用设备: {DEVICE}")

当前工作目录: E:\student\模型
 配置完成，使用设备: cuda


In [6]:
# 载入模型
model = CodeBertDetector(
    model_path=LOCAL_MODEL_PATH,
    num_types=len(TYPE_LABELS),
    dropout_prob=0.1,
    max_length=256,
    load_tokenizer=True
)

if CHECKPOINT_PATH.exists():
    state_dict = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(state_dict)
    print(" 模型载入成功")
else:
    print(f" 未找到模型文件: {CHECKPOINT_PATH}")

model.to(DEVICE)
model.eval()

# 载入测试集
with open(TEST_DATA_PATH, 'r', encoding='utf-8') as f:
    test_data = json.load(f)
    
print(f" 测试集载入成功，共 {len(test_data)} 个样本")

C:\Users\bht20\miniconda3\envs\dlgpu\lib\site-packages\transformers\tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


✅ 模型载入成功
✅ 测试集载入成功，共 389 个样本


In [7]:
# 随机抽取样本进行演示
NUM_SAMPLES = 5
samples = random.sample(test_data, min(NUM_SAMPLES, len(test_data)))

results = []
tokenizer = model.tokenizer

for sample in samples:
    code = sample.get("code", "")
    true_binary = sample.get("binary_label", sample.get("label", 0))
    
    # 编码
    encoded = tokenizer(
        code,
        truncation=True,
        padding="max_length",
        max_length=256,
        return_tensors="pt"
    )
    
    input_ids = encoded["input_ids"].to(DEVICE)
    attention_mask = encoded["attention_mask"].to(DEVICE)
    
    # 预测
    with torch.no_grad():
        outputs = model.predict(input_ids, attention_mask, return_probs=True)
    
    binary_probs = outputs["binary_probs"][0].cpu().numpy()
    type_probs = outputs["type_probs"][0].cpu().numpy()
    binary_pred = outputs["binary_pred"][0].cpu().item()
    type_pred = outputs["type_pred"][0].cpu().item()
    
    results.append({
        "code": code[:200] + "..." if len(code) > 200 else code,
        "true_binary": true_binary,
        "pred_binary": binary_pred,
        "pred_type": type_pred,
        "binary_confidence": binary_probs[binary_pred],
        "type_confidence": type_probs[type_pred],
        "safe_prob": binary_probs[0],
        "malicious_prob": binary_probs[1]
    })

print(f" 完成 {len(results)} 个样本的预测")

 完成 5 个样本的预测


C:\Users\bht20\miniconda3\envs\dlgpu\lib\site-packages\transformers\models\roberta\modeling_roberta.py:370: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


In [8]:
# 显示预测结果
correct = 0

for i, result in enumerate(results):
    print("=" * 80)
    print(f"样本 #{i+1}")
    print("=" * 80)
    
    # 显示代码片段
    print(f"\n📝 代码片段:\n{result['code']}\n")
    
    # 预测结果
    binary_label = "恶意" if result["pred_binary"] == 1 else " 安全"
    true_label = "恶意" if result["true_binary"] == 1 else " 安全"
    is_correct = result["pred_binary"] == result["true_binary"]
    correct += is_correct
    
    print(f"预测: {binary_label} (置信度: {result['binary_confidence']:.2%}) {'✅' if is_correct else '❌'}")
    print(f"   真实: {true_label}")
    
    # 类型预测
    if result["pred_binary"] == 1:
        type_name = TYPE_LABELS.get(result["pred_type"], "未知")
        print(f"   类型: {type_name} (置信度: {result['type_confidence']:.2%})")
    
    print(f"\n 概率: 安全 {result['safe_prob']:.2%} | 恶意 {result['malicious_prob']:.2%}")
    print()

print("=" * 80)
print(f" 统计: 准确率 {correct}/{len(results)} = {correct/len(results):.2%}")
print("=" * 80)

样本 #1

📝 代码片段:
<?php

function x()
{

    return "/*sasas23123*/".$_POST['a']."/*sdfw3123*/";

}

eval(x());
?>


预测: 恶意 (置信度: 98.56%) ✅
   真实: 恶意
   类型: WebShell (置信度: 67.29%)

 概率: 安全 1.44% | 恶意 98.56%

样本 #2

📝 代码片段:
[
    {
        "id": 116205741,
        "name": "CVE-2017-17692",
        "full_name": "specloli\/CVE-2017-17692",
        "owner": {
            "login": "specloli",
            "id": 10647366,
    ...

预测: 恶意 (置信度: 99.35%) ✅
   真实: 恶意
   类型: OtherMalicious (置信度: 83.23%)

 概率: 安全 0.65% | 恶意 99.35%

样本 #3

📝 代码片段:
[
    {
        "id": 89426502,
        "name": "Exploit-Development",
        "full_name": "wetw0rk\/Exploit-Development",
        "owner": {
            "login": "wetw0rk",
            "id": 2076354...

预测: 恶意 (置信度: 99.31%) ✅
   真实: 恶意
   类型: OtherMalicious (置信度: 83.73%)

 概率: 安全 0.69% | 恶意 99.31%

样本 #4

📝 代码片段:
//VAMOS CRIAR UM PROGRAMA QUE TESTA AS ROTINAS CRIADAS NO ARQUIVO prime_power_detection.h
//COMPILAR ESTE PROGRAMA COM O COMANDO: gcc -o prime_pow